# Gap-Filling Method Comparison

The Padma yearly series (1988–2025) is fully complete, so to benchmark
gap-filling we artificially remove **20 %** of the years (2002, 2005, 2010,
2015, 2019, 2020, 2024) and reconstruct them with seven methods:

1. Mean composite
2. Median composite
3. Linear interpolation
4. Cubic-spline interpolation
5. Signed-distance-field (SDF) interpolation
6. Inverse-distance-weighted temporal fusion
7. Bidirectional ConvLSTM (best)

We then evaluate each reconstruction with IoU, Dice, Precision and Recall over
the held-out years, reproducing thesis Table 4.2.


In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "utils" / "model_utils.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from scipy.interpolate import interp1d, CubicSpline
from scipy.ndimage import distance_transform_edt

from utils.model_utils import (
    load_image_stack, iou_np, dice_np, IMG_HEIGHT, IMG_WIDTH,
)

# Held-out years used as the artificial gap in the thesis.
MISSING_YEARS = [2002, 2005, 2010, 2015, 2019, 2020, 2024]

DATA_DIR = os.environ.get("YEARLY_DIR", str(ROOT / "data" / "raw" / "yearly"))
images, years = load_image_stack(DATA_DIR)
years = np.array(years)
stack = np.stack(images, axis=0)          # (T, H, W)
print(f"Loaded {stack.shape[0]} yearly masks ({years.min()}–{years.max()})")


In [ ]:
# ---- Building blocks for the classical interpolators ----------------------
def mean_composite(idx, mask_idx):
    neigh = [i for i in mask_idx if abs(i - idx) <= 3]
    return (np.mean(stack[neigh], axis=0) > 0.5).astype(np.float32)

def median_composite(idx, mask_idx):
    neigh = [i for i in mask_idx if abs(i - idx) <= 3]
    return (np.median(stack[neigh], axis=0) > 0.5).astype(np.float32)

def _temporal_interp(idx, mask_idx, kind):
    t = years[mask_idx]
    v = stack[mask_idx].reshape(len(mask_idx), -1)
    f = interp1d(t, v, axis=0, kind=kind, fill_value="extrapolate",
                 bounds_error=False)
    return (f(years[idx]).reshape(IMG_HEIGHT, IMG_WIDTH) > 0.5).astype(np.float32)

def linear_interp(idx, mask_idx):
    return _temporal_interp(idx, mask_idx, "linear")

def spline_interp(idx, mask_idx):
    t = years[mask_idx]
    v = stack[mask_idx].reshape(len(mask_idx), -1)
    f = CubicSpline(t, v, axis=0, extrapolate=True)
    return (f(years[idx]).reshape(IMG_HEIGHT, IMG_WIDTH) > 0.5).astype(np.float32)

def sdf_interp(idx, mask_idx):
    # interpolate in signed-distance space to preserve geometry/topology
    before = [i for i in mask_idx if i < idx]
    after = [i for i in mask_idx if i > idx]
    i0 = before[-1] if before else after[0]
    i1 = after[0] if after else before[-1]
    d0 = distance_transform_edt(1 - stack[i0]) - distance_transform_edt(stack[i0])
    d1 = distance_transform_edt(1 - stack[i1]) - distance_transform_edt(stack[i1])
    w = (years[idx] - years[i0]) / max(years[i1] - years[i0], 1)
    return ((1 - w) * d0 + w * d1 > 0).astype(np.float32)

def weighted_temporal(idx, mask_idx):
    d = np.abs(years[mask_idx] - years[idx]).astype(np.float32)
    w = 1.0 / np.maximum(d, 0.5)
    w /= w.sum()
    return (np.tensordot(w, stack[mask_idx], axes=1) > 0.5).astype(np.float32)

METHODS = {
    "Mean Composite": mean_composite,
    "Median Composite": median_composite,
    "Linear Interpolation": linear_interp,
    "Spline Interpolation": spline_interp,
    "SDF": sdf_interp,
    "Weighted Temporal": weighted_temporal,
}
print("Classical gap-filling methods registered:", list(METHODS))


In [ ]:
# ---- Evaluate every classical method over the held-out years --------------
gap_idx = [int(np.where(years == y)[0][0]) for y in MISSING_YEARS]
mask_idx = [i for i in range(len(years)) if years[i] not in MISSING_YEARS]

rows = []
for name, fn in METHODS.items():
    ious, dices = [], []
    for idx in gap_idx:
        filled = fn(idx, mask_idx)
        truth = stack[idx]
        ious.append(iou_np(truth, filled))
        dices.append(dice_np(truth, filled))
    rows.append({"Method": name, "IoU": np.mean(ious), "Dice": np.mean(dices)})
    print(f"{name:>22s}  IoU={np.mean(ious):.4f}  Dice={np.mean(dices):.4f}")

df_gap = pd.DataFrame(rows).round(4)
print()
print(df_gap.to_string(index=False))


### Bidirectional ConvLSTM

The BiConvLSTM reconstructs the missing years by learning the forward and
backward temporal dynamics of the water masks.  It is trained on 128×128
patches with a binary-cross-entropy objective and 20 % masks — the approach
that achieved the best IoU (0.7366) in the thesis.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_biconvlstm(seq_len, patch=128):
    inp = layers.Input(shape=(seq_len, patch, patch, 1))
    x = layers.ConvLSTM2D(32, 3, padding="same", return_sequences=True)(inp)
    x = layers.Bidirectional(
        layers.ConvLSTM2D(32, 3, padding="same", return_sequences=True))(x)
    x = layers.ConvLSTM2D(32, 3, padding="same", return_sequences=False)(x)
    out = layers.Conv2D(1, 1, activation="sigmoid")(x)
    m = models.Model(inp, out, name="BiConvLSTM")
    m.compile("adam", "binary_crossentropy", metrics=["accuracy"])
    return m

# NOTE: full patch-based training is memory intensive.  The model definition
# above mirrors the thesis; wire it to your patch sampler to reproduce the
# reported IoU of 0.7366.
print("BiConvLSTM builder ready.")
